In [0]:
%sql
SELECT * FROM np_binary_shap

In [0]:
%pip install xgboost
%restart_python

# Train and Test split between both tables

In [0]:
from pyspark.sql import functions as F

# Full Datasets
# Living: multimodal_adni_living_model_ready_SHAP_no_MOCA
# Deceased: NP_Binary_SHAP

# 50/50 Datasets  
# Living: alzheimers_balanced_small
# Deceased: NP_Binary_SHAP_50_50

train_table = "alzheimers_balanced_small"
test_table = "NP_Binary_SHAP_50_50"

df_train = spark.table(train_table)
df_test = spark.table(test_table)

In [0]:
df = spark.table("NP_Binary_SHAP_50_50")
pdf = df.toPandas()

# Save to driver local storage
local_path = "/dbfs/NP_Binary_SHAP_50_50.csv"
pdf.to_csv(local_path, index=False)

print(local_path)

In [0]:
print((df_test.count(), len(df_test.columns)))

In [0]:
pdf_train = df_train.toPandas()
pdf_test = df_test.toPandas()

In [0]:
TARGET = "y_mmse_binary"

# Check to make sure columns are the same
feature_cols = [c for c in pdf_train.columns if c != TARGET]

X_train = pdf_train[feature_cols]
y_train = pdf_train[TARGET]

X_test = pdf_test[feature_cols]
y_test = pdf_test[TARGET]

Full Dataset Shapes:

Train shape: (4537, 19)
Test shape: (121, 19)

50-50 Dataset Shapes:

Train shape: (864, 19)
Test shape: (46, 19)

In [0]:
print("Train shape:", pdf_train.shape)
print("Test shape:", pdf_test.shape)

In [0]:
# All Imports for models

# Shared
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score


# RF + All
from sklearn.ensemble import RandomForestClassifier


# XGB
from xgboost import XGBClassifier

# SVM
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from mlflow.models.signature import infer_signature

## Train Model (Random Forest)

In [0]:
# Base model
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# Hyperparameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

# Grid search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

with mlflow.start_run(run_name="RF_GridSearch"):

    grid_search.fit(X_train, y_train)

    rf_best_model = grid_search.best_estimator_

    # Probability predictions
    y_train_pred_proba = rf_best_model.predict_proba(X_train)[:, 1]
    y_test_pred_proba = rf_best_model.predict_proba(X_test)[:, 1]

    # Class predictions
    y_train_pred_class = rf_best_model.predict(X_train)
    y_test_pred_class = rf_best_model.predict(X_test)

    # Create prediction table
    rf_test_predictions = pd.DataFrame({
        "row_id": X_test.index,
        "actual": y_test,
        "predicted_class": y_test_pred_class,
        "predicted_probability": y_test_pred_proba,
        "correct": y_test == y_test_pred_class
    })

    mlflow.log_table(rf_test_predictions, "predictions/rf_test_predictions.json")

    # AUC
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    auc_gap = train_auc - test_auc

    # Other Metrics
    train_pr = average_precision_score(y_train, y_train_pred_proba)
    test_pr = average_precision_score(y_test, y_test_pred_proba)
    train_accuracy = accuracy_score(y_train, y_train_pred_class)
    test_accuracy = accuracy_score(y_test, y_test_pred_class)
    train_precision = precision_score(y_train, y_train_pred_class)
    test_precision = precision_score(y_test, y_test_pred_class)
    train_recall = recall_score(y_train, y_train_pred_class)
    test_recall = recall_score(y_test, y_test_pred_class)
    train_f1 = f1_score(y_train, y_train_pred_class)
    test_f1 = f1_score(y_test, y_test_pred_class)

    # Log best params
    mlflow.log_params(grid_search.best_params_)

    # Log metrics
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("auc_gap", auc_gap)
    mlflow.log_metric("train_pr_auc", train_pr)
    mlflow.log_metric("test_pr_auc", test_pr)
    mlflow.log_metric("best_cv_auc", grid_search.best_score_)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("test_recall", test_recall)
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("test_f1", test_f1)

    mlflow.sklearn.log_model(rf_best_model, "best_model")

print("Best Params:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)
print("AUC Gap:", auc_gap)
print("Train PR AUC:", train_pr)
print("Test PR AUC:", test_pr)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train Precision:", train_precision)
print("Test Precision:", test_precision)
print("Train Recall:", train_recall)
print("Test Recall:", test_recall)
print("Train F1:", train_f1)
print("Test F1:", test_f1)

%md
## Train Model (XGBoost)

In [0]:
# Base model
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=1
)

# Hyperparameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# Grid search
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

with mlflow.start_run(run_name="XGB_GridSearch"):

    grid_search.fit(X_train, y_train)

    xgb_best_model = grid_search.best_estimator_

    # Probability predictions
    y_train_pred_proba = xgb_best_model.predict_proba(X_train)[:, 1]
    y_test_pred_proba = xgb_best_model.predict_proba(X_test)[:, 1]

    # Class predictions
    y_train_pred_class = xgb_best_model.predict(X_train)
    y_test_pred_class = xgb_best_model.predict(X_test)

    # Create prediction table
    xgb_test_predictions = pd.DataFrame({
        "row_id": X_test.index,
        "actual": y_test,
        "predicted_class": y_test_pred_class,
        "predicted_probability": y_test_pred_proba,
        "correct": y_test == y_test_pred_class
    })

    mlflow.log_table(xgb_test_predictions, "predictions/xgb_test_predictions.json")

    # AUC
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    auc_gap = train_auc - test_auc

    # Other metrics
    train_pr = average_precision_score(y_train, y_train_pred_proba)
    test_pr = average_precision_score(y_test, y_test_pred_proba)

    train_accuracy = accuracy_score(y_train, y_train_pred_class)
    test_accuracy = accuracy_score(y_test, y_test_pred_class)

    train_precision = precision_score(y_train, y_train_pred_class)
    test_precision = precision_score(y_test, y_test_pred_class)

    train_recall = recall_score(y_train, y_train_pred_class)
    test_recall = recall_score(y_test, y_test_pred_class)

    train_f1 = f1_score(y_train, y_train_pred_class)
    test_f1 = f1_score(y_test, y_test_pred_class)

    # Log best params
    mlflow.log_params(grid_search.best_params_)

    # Log metrics
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("auc_gap", auc_gap)
    mlflow.log_metric("train_pr_auc", train_pr)
    mlflow.log_metric("test_pr_auc", test_pr)
    mlflow.log_metric("best_cv_auc", grid_search.best_score_)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("test_recall", test_recall)
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("test_f1", test_f1)

    mlflow.xgboost.log_model(xgb_best_model, "best_model")

print("Best Params:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)
print("AUC Gap:", auc_gap)
print("Train PR AUC:", train_pr)
print("Test PR AUC:", test_pr)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train Precision:", train_precision)
print("Test Precision:", test_precision)
print("Train Recall:", train_recall)
print("Test Recall:", test_recall)
print("Train F1:", train_f1)
print("Test F1:", test_f1)

## Train Model (SVM)

In [0]:
# Pipeline: impute -> scale -> SVM
svm_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svm", SVC(probability=True, random_state=42))
])

# Reasonable grid size
param_grid = {
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.001]
}

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

with mlflow.start_run(run_name="SVM_GridSearch"):

    grid_search.fit(X_train, y_train)

    svm_best_model = grid_search.best_estimator_

    # Probability predictions
    y_train_pred_proba = svm_best_model.predict_proba(X_train)[:, 1]
    y_test_pred_proba = svm_best_model.predict_proba(X_test)[:, 1]

    # Class predictions
    y_train_pred_class = svm_best_model.predict(X_train)
    y_test_pred_class = svm_best_model.predict(X_test)

    # Create prediction table
    svm_test_predictions = pd.DataFrame({
        "row_id": X_test.index,
        "actual": y_test,
        "predicted_class": y_test_pred_class,
        "predicted_probability": y_test_pred_proba,
        "correct": y_test == y_test_pred_class
    })

    mlflow.log_table(svm_test_predictions, "predictions/svm_test_predictions.json")

    # AUC
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    auc_gap = train_auc - test_auc

    # Other Metrics
    train_pr = average_precision_score(y_train, y_train_pred_proba)
    test_pr = average_precision_score(y_test, y_test_pred_proba)
    train_accuracy = accuracy_score(y_train, y_train_pred_class)
    test_accuracy = accuracy_score(y_test, y_test_pred_class)
    train_precision = precision_score(y_train, y_train_pred_class)
    test_precision = precision_score(y_test, y_test_pred_class)
    train_recall = recall_score(y_train, y_train_pred_class)
    test_recall = recall_score(y_test, y_test_pred_class)
    train_f1 = f1_score(y_train, y_train_pred_class)
    test_f1 = f1_score(y_test, y_test_pred_class)

    # Log best params
    mlflow.log_params(grid_search.best_params_)

    # Log metrics
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("auc_gap", auc_gap)
    mlflow.log_metric("train_pr_auc", train_pr)
    mlflow.log_metric("test_pr_auc", test_pr)
    mlflow.log_metric("best_cv_auc", grid_search.best_score_)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("test_recall", test_recall)
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("test_f1", test_f1)

    mlflow.sklearn.log_model(svm_best_model, "best_model")

print("Best Params:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)
print("AUC Gap", auc_gap)
print("Train PR AUC:", train_pr)
print("Test PR AUC:", test_pr)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train Precision:", train_precision)
print("Test Precision:", test_precision)
print("Train Recall:", train_recall)
print("Test Recall:", test_recall)
print("Train F1:", train_f1)
print("Test F1:", test_f1)

## Evaluate Each Model

In [0]:
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Probabilities (needed for ROC-AUC)
    if hasattr(model, "predict_proba"):
        y_test_prob = model.predict_proba(X_test)[:, 1]
    else:
        # fallback (rare case)
        y_test_prob = model.decision_function(X_test)

    # Metrics
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_test_pred),
        "Precision": precision_score(y_test, y_test_pred),
        "Recall": recall_score(y_test, y_test_pred),
        "F1 Score": f1_score(y_test, y_test_pred),
        "ROC-AUC": roc_auc_score(y_test, y_test_prob)
    }

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"]
    )

    print(f"\n{name} Results:")
    for k, v in metrics.items():
        if k != "Model":
            print(f"{k}: {v:.4f}")

    print("\nConfusion Matrix:")
    display(cm_df)
    return metrics, cm_df

In [0]:
rf_metrics, rf_cm = evaluate_model("Random Forest", rf_best_model, X_train, y_train, X_test, y_test)

xgb_metrics, xgb_cm = evaluate_model("XGBoost", xgb_best_model, X_train, y_train, X_test, y_test)

svm_metrics, svm_cm = evaluate_model("SVM", svm_best_model, X_train, y_train, X_test, y_test)

In [0]:
# Show condensed results from each model as a table
results_df = pd.DataFrame([rf_metrics, xgb_metrics, svm_metrics])
print(results_df)

## Full Dataset Stress Test
Full dataset with stresstest has very poor recall rates, learns the patterns of non-AD very well, but performs poorly to pick out those without AD.

![image_1777155637685.png](./image_1777155637685.png "image_1777155637685.png")

## 50-50 Dataset Stress Test
Much better results, similar to those of the actual testing on living/non-deceased. Recall issue is fixed, but has a minor issue with False Positives. Might benefit from a 60/40 Balance or something similar.

![image_1777155841857.png](./image_1777155841857.png "image_1777155841857.png")

## Add Predictions as artifacts
I noticed that both RF and XGB have 2-21 for positives, so I wanted to check if these were all the same incorrect predictions.

```python
test_predictions = pd.DataFrame({
    "row_id": X_test.index,
    "actual": y_test,
    "predicted_class": y_test_pred_class,
    "predicted_probability": y_test_pred_proba,
    "correct": y_test == y_test_pred_class
})

mlflow.log_table(_test_predictions, "predictions/_test_predictions.json")
```

After doing some snooping around the predictions. I was correct... ALL of the models miss the exact same entries, index 7 and 38.

- Interestingly, ONE of the entries is really close; the tree models gave it 40% of AD, while SVM was much lower probability. (Index 38, actual AD, predicted non-AD)
  - Random Forest: 43.2% (0.4321620604)
  - XGBoost: 40.4% (0.4040622413)
  - SVM: 16.3% (0.1630290683)

- The other entry that was incorrect for all three models was given an extremely low probability. (Index 7, actual AD, predicted non-AD)
  - Random Forest: 10.6% (0.1063805168)
  - XGBoost: 9.2% (0.0918186903)
  - SVM: 2.7% (0.0268341249)